> **Development implementation; end-to-end reproduction not verified.** See [reproduction notes](../docs/reproduction.md) and the [release checklist](../docs/release-checklist.md). Cleared outputs are not evidence of a successful run.

# 02 · Time-varying electricity-background acquisition

This notebook contains the ENTSO-E acquisition workflow used during development
of the KTB1 demonstration of the dynamic LCA framework. It requires your own
`ENTSOE_API_KEY` environment variable and network access. Set the variable outside
the notebook before starting Jupyter; never paste it into a committed cell.

**The source date windows and resampling have been retained, not validated.**
In particular, the annual query ends at 2025-08-10 00:00 and the supplied analysis
outputs showed 8,736 hours. Confirm the intended calendar before final extraction.
Running this notebook contacts ENTSO-E and writes local CSV files; it is not a
bundled immutable input dataset. Fresh API results may differ from historical data.

Generation and price files default to `data/local/`; the optional joined export
goes to `results/`. Environment overrides are defined below. A `.env` file is not
loaded automatically.


In [ ]:
# Local configuration: no credentials are stored or printed by this cell.
import os
from pathlib import Path

def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter from the repository or notebooks directory.")

REPO_ROOT = _find_repo_root(Path.cwd().resolve())
INPUT_DIR = Path(os.environ.get("DLCA_INPUT_DIR", str(REPO_ROOT / "data" / "local"))).expanduser()
OUTPUT_DIR = Path(os.environ.get("DLCA_OUTPUT_DIR", str(REPO_ROOT / "results"))).expanduser()
MODEL_DIR = Path(os.environ.get("DLCA_MODEL_DIR", str(REPO_ROOT / "model" / "local"))).expanduser()
MODEL_NAME = os.environ.get("DLCA_MODEL_NAME", "KTB1_DLCA1_newvariable_22_legacy")
GENERATION_CSV = Path(os.environ.get("DLCA_GENERATION_CSV", str(INPUT_DIR / "dk2_generation_2024-08-11_2025-08-10.csv"))).expanduser()
PRICE_CSV = Path(os.environ.get("DLCA_PRICE_CSV", str(INPUT_DIR / "dk2_price_2024-08-11_2025-08-10.csv"))).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Libraries Import

In [ ]:
#Electricity type:
import requests
import pandas as pd
from io import StringIO

## ENTSOe Websites and docs

In [ ]:
#https://transparency.entsoe.eu/dashboard/show
# ELECTRICITY TYPE:
#https://transparency.entsoe.eu/generation/r2/actualGenerationPerProductionType/show?name=&defaultValue=false&viewType=GRAPH&areaType=BZN&atch=false&datepicker-day-offset-select-dv-date-from_input=D&dateTime.dateTime=13.08.2025+00:00|CET|DAYTIMERANGE&dateTime.endDateTime=13.08.2025+00:00|CET|DAYTIMERANGE&area.values=CTY|10Y1001A1001A65H!BZN|10YDK-2--------M&productionType.values=B01&productionType.values=B02&productionType.values=B03&productionType.values=B04&productionType.values=B05&productionType.values=B06&productionType.values=B07&productionType.values=B08&productionType.values=B09&productionType.values=B10&productionType.values=B11&productionType.values=B12&productionType.values=B13&productionType.values=B14&productionType.values=B20&productionType.values=B15&productionType.values=B16&productionType.values=B17&productionType.values=B18&productionType.values=B19&dateTime.timezone=CET_CEST&dateTime.timezone_input=CET+(UTC+1)+/+CEST+(UTC+2)
# DAY-AHEAD PRICE:
#https://transparency.entsoe.eu/transmission-domain/r2/dayAheadPrices/show?areaType=BZN&atch=false&dateTime.dateTime=&defaultValue=false&name=&viewType=TABLE&utm_source=chatgpt.com

#log in to the ENTSO‑E Transparency Platform with the account you registered. 
#Under “My Account Settings”, a section labelled “Web API Security Token.” Generate or copy your personal token. 
#The key is then visible in your ENTSO‑E account under ‘Web API Security Token
#Entso-e API security token XXX

## Installation & Imports

In [ ]:
#pip install entsoe-py

In [ ]:
from entsoe import EntsoePandasClient
client = EntsoePandasClient(api_key=os.environ["ENTSOE_API_KEY"])


## Energy Type

In [ ]:
# example: fetch total generation and load for Denmark on one day
start = pd.Timestamp('2025-01-01', tz='Europe/Paris')
end   = pd.Timestamp('2025-01-02', tz='Europe/Paris')

generation = client.query_generation(country_code='DK', start=start, end=end)
load       = client.query_load(country_code='DK', start=start, end=end)

In [ ]:
# Inspect the results
print(generation.head())
print(load.head())

In [ ]:
from entsoe import mappings
print(mappings.PSRTYPE_MAPPINGS)

In [ ]:
# Full-year period 
start = pd.Timestamp('2024-08-11 00:00', tz='Europe/Paris')
end   = pd.Timestamp('2025-08-10 00:00', tz='Europe/Paris')

df = client.query_generation(country_code='DK_2', start=start, end=end)
df = df.tz_convert('Europe/Paris').resample('h').sum()

# Preview the first five hours
print(df.head(5))

In [ ]:
# Save generation locally; this does not publish data.
save_path = GENERATION_CSV
save_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(save_path)


## Price

In [ ]:
start = pd.Timestamp('2025-01-01', tz='Europe/Paris')
end   = pd.Timestamp('2025-01-02', tz='Europe/Paris')

prices = client.query_day_ahead_prices(country_code='DK_2', start=start, end=end)
prices = prices.tz_convert('Europe/Paris').resample('h').mean().rename('DA_price_EUR_MWh')

print(prices.head())

In [ ]:
print("Timezone:", prices.index.tz)
print("Count:", prices.size, "| Range:", prices.index.min(), "→", prices.index.max())

In [ ]:
DK2_EIC = "10YDK-2--------M"
print("DK2 EIC (bidding zone):", DK2_EIC)

In [ ]:
start = pd.Timestamp('2024-08-11 00:00', tz='Europe/Paris')
end   = pd.Timestamp('2025-08-10 00:00', tz='Europe/Paris')

pr = client.query_day_ahead_prices(country_code='DK_2', start=start, end=end)
pr = pr.tz_convert('Europe/Paris').resample('h').mean().rename('DA_price_EUR_MWh')

print(pr.head(5))

In [ ]:
save_price = PRICE_CSV
save_price.parent.mkdir(parents=True, exist_ok=True)
pr.to_csv(save_price)
print("✓ Saved:", save_price)

# OPTIONAL: join prices to your generation df (from your step 8)
# df = ... (your generation table, hourly 'Europe/Paris')
df_join = df.join(pr, how='left')
save_both = OUTPUT_DIR / "dk2_generation_and_price_2024-08-11_2025-08-10.csv"
df_join.to_csv(save_both)
print("✓ Saved:", save_both)